# SmartBlood — Multi-Risk Inventory Engine

This notebook integrates the trained XGBoost demand forecasting model
with a multi-risk inventory decision-support engine.

The system assesses:

1. Shortage Risk
2. Excess Stock Risk
3. Expiry Risk

The final output is designed to support the SmartBlood hospital
management dashboard.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

In [2]:
# =========================================================
# CONFIGURATION
# =========================================================

# Paths
MODEL_PATH = "../outputs/models/smartblood_xgboost_model.pkl"

DATA_PATH = "../data/features/blood_demand_features.csv"

# Forecast planning horizon
HORIZON_DAYS = 7


# =========================================================
# PROTOTYPE RISK THRESHOLDS
# These are configurable prototype values.
# =========================================================

SHORTAGE_HIGH_THRESHOLD = 15
EXCESS_HIGH_THRESHOLD = 50
EXPIRY_HIGH_THRESHOLD = 20

In [3]:
# =========================================================
# LOAD TRAINED XGBOOST MODEL
# =========================================================

model = joblib.load(MODEL_PATH)

print("Model loaded successfully!")
print("Model type:", type(model).__name__)

Model loaded successfully!
Model type: XGBRegressor


In [4]:
# =========================================================
# GET MODEL FEATURE COLUMNS
# =========================================================

if hasattr(model, "feature_names_in_"):
    
    feature_columns = list(model.feature_names_in_)
    
    print("Number of model features:", len(feature_columns))
    
    print("\nFeatures used by the model:")
    for feature in feature_columns:
        print("-", feature)

else:
    raise ValueError(
        "The loaded model does not contain feature_names_in_. "
        "Use the feature columns from your forecasting notebook."
    )

Number of model features: 24

Features used by the model:
- demand
- demand_lag_1
- demand_lag_7
- demand_lag_14
- demand_lag_30
- demand_rolling_7
- demand_rolling_14
- demand_rolling_30
- emergency_requests_lag_1
- target_day_of_week_num
- target_month
- target_quarter
- target_is_weekend
- blood_group_A+
- blood_group_A-
- blood_group_AB+
- blood_group_AB-
- blood_group_B+
- blood_group_B-
- blood_group_O+
- blood_group_O-
- component_Plasma
- component_Platelets
- component_RBC


In [7]:
# =========================================================
# LOAD FEATURE-ENGINEERED DATA
# =========================================================

df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

df = df.sort_values(
    ["blood_group", "component", "date"]
).reset_index(drop=True)

print("Feature dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Feature dataset loaded successfully!
Dataset shape: (16800, 35)


,date,blood_group,component,demand,collected,issued,opening_stock,closing_stock,expired,emergency,...,emergency_risk,demand_lag_1,demand_lag_7,demand_rolling_7,demand_lag_14,demand_lag_30,demand_rolling_14,demand_rolling_30,emergency_requests_lag_1,target_demand
0,2023-01-31,A+,Plasma,29,32,25,145,152,0,2,...,1,23.0,28.0,30.000000,41.0,32.0,28.428571,30.800000,3.0,30.0
1,2023-02-01,A+,Plasma,30,24,29,152,147,0,3,...,1,29.0,30.0,30.142857,26.0,35.0,27.571429,30.700000,2.0,25.0
2,2023-02-02,A+,Plasma,25,18,25,147,140,0,1,...,1,30.0,32.0,30.142857,25.0,23.0,27.857143,30.533333,3.0,26.0
3,2023-02-03,A+,Plasma,26,25,25,140,140,0,1,...,1,25.0,28.0,29.142857,21.0,52.0,27.857143,30.600000,1.0,21.0
4,2023-02-04,A+,Plasma,21,27,21,140,145,1,1,...,1,26.0,32.0,28.857143,26.0,32.0,28.214286,29.733333,1.0,25.0


In [8]:
# =========================================================
# CHECK DATASET COLUMNS
# =========================================================

print("Dataset columns:\n")

for column in df.columns:
    print("-", column)

Dataset columns:

- date
- blood_group
- component
- demand
- collected
- issued
- opening_stock
- closing_stock
- expired
- emergency
- day_of_week
- day_of_week_num
- month
- month_name
- year
- quarter
- is_weekend
- blood_product
- calculated_closing_stock
- stock_difference
- expiry_rate
- shortage_risk
- stock_coverage_ratio
- excess_stock_risk
- expiry_risk
- emergency_risk
- demand_lag_1
- demand_lag_7
- demand_rolling_7
- demand_lag_14
- demand_lag_30
- demand_rolling_14
- demand_rolling_30
- emergency_requests_lag_1
- target_demand


In [9]:
# =========================================================
# SELECT BLOOD PRODUCT
# =========================================================

SELECTED_BLOOD_GROUP = "O+"
SELECTED_COMPONENT = "RBC"

print(
    "Selected product:",
    SELECTED_BLOOD_GROUP,
    "-",
    SELECTED_COMPONENT
)

Selected product: O+ - RBC


In [10]:
# =========================================================
# CHECK AVAILABLE VALUES
# =========================================================

print("Available blood groups:")
print(sorted(df["blood_group"].unique()))

print("\nAvailable components:")
print(sorted(df["component"].unique()))

Available blood groups:
['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']

Available components:
['Plasma', 'Platelets', 'RBC']


In [11]:
# =========================================================
# FILTER PRODUCT HISTORY
# =========================================================

product_history = df[
    (df["blood_group"] == SELECTED_BLOOD_GROUP)
    &
    (df["component"] == SELECTED_COMPONENT)
].copy()

product_history = product_history.sort_values(
    "date"
).reset_index(drop=True)

print("Number of historical records:", len(product_history))

print("\nLast available records:")
display(product_history.tail())

Number of historical records: 700

Last available records:


,date,blood_group,component,demand,collected,issued,opening_stock,closing_stock,expired,emergency,...,emergency_risk,demand_lag_1,demand_lag_7,demand_rolling_7,demand_lag_14,demand_lag_30,demand_rolling_14,demand_rolling_30,emergency_requests_lag_1,target_demand
695,2024-12-26,O+,RBC,62,82,61,1542,1563,0,2,...,1,61.0,72.0,58.857143,61.0,58.0,59.357143,57.633333,4.0,55.0
696,2024-12-27,O+,RBC,55,54,54,1563,1563,0,2,...,1,62.0,54.0,57.428571,60.0,50.0,59.428571,57.766667,2.0,59.0
697,2024-12-28,O+,RBC,59,70,60,1563,1573,0,2,...,1,55.0,62.0,57.571429,61.0,52.0,59.071429,57.933333,2.0,59.0
698,2024-12-29,O+,RBC,59,47,52,1573,1568,0,1,...,1,59.0,54.0,57.142857,62.0,52.0,58.928571,58.166667,2.0,67.0
699,2024-12-30,O+,RBC,67,71,61,1568,1578,0,8,...,1,59.0,53.0,57.857143,62.0,72.0,58.714286,58.400000,1.0,57.0


In [12]:
# =========================================================
# VALIDATE HISTORY
# =========================================================

if len(product_history) < 31:
    
    raise ValueError(
        "At least 31 historical records are required "
        "for a 30-day lag forecast."
    )

print("Historical data validation successful!")

print(
    "Last historical date:",
    product_history["date"].max()
)

Historical data validation successful!
Last historical date: 2024-12-30 00:00:00


In [13]:
# =========================================================
# FIND EMERGENCY COLUMN
# =========================================================

possible_emergency_columns = [
    "emergency_requests",
    "emergency"
]

emergency_column = None

for column in possible_emergency_columns:
    
    if column in product_history.columns:
        emergency_column = column
        break


if emergency_column is None:
    
    raise ValueError(
        "Emergency-related column was not found."
    )


print(
    "Emergency column being used:",
    emergency_column
)

Emergency column being used: emergency


In [14]:
# =========================================================
# BUILD ONE FUTURE FEATURE ROW
# =========================================================

def build_future_feature_row(
    demand_history,
    future_date,
    selected_blood_group,
    selected_component,
    emergency_value,
    feature_columns
):
    
    # Start with all model features set to zero
    future_row = {
        feature: 0
        for feature in feature_columns
    }
    
    
    # =====================================================
    # DEMAND FEATURES
    # =====================================================
    
    # Use the latest available demand
    if "demand" in future_row:
        future_row["demand"] = demand_history[-1]
    
    
    # Lag features
    if "demand_lag_1" in future_row:
        future_row["demand_lag_1"] = demand_history[-1]
    
    if "demand_lag_7" in future_row:
        future_row["demand_lag_7"] = demand_history[-7]
    
    if "demand_lag_14" in future_row:
        future_row["demand_lag_14"] = demand_history[-14]
    
    if "demand_lag_30" in future_row:
        future_row["demand_lag_30"] = demand_history[-30]
    
    
    # =====================================================
    # ROLLING FEATURES
    # =====================================================
    
    if "demand_rolling_7" in future_row:
        future_row["demand_rolling_7"] = np.mean(
            demand_history[-7:]
        )
    
    if "demand_rolling_14" in future_row:
        future_row["demand_rolling_14"] = np.mean(
            demand_history[-14:]
        )
    
    if "demand_rolling_30" in future_row:
        future_row["demand_rolling_30"] = np.mean(
            demand_history[-30:]
        )
    
    
    # =====================================================
    # EMERGENCY FEATURE
    # =====================================================
    
    if "emergency_requests_lag_1" in future_row:
        
        future_row[
            "emergency_requests_lag_1"
        ] = emergency_value
    
    
    # =====================================================
    # CALENDAR FEATURES
    # =====================================================
    
    if "target_day_of_week_num" in future_row:
        future_row[
            "target_day_of_week_num"
        ] = future_date.dayofweek
    
    
    if "target_month" in future_row:
        future_row[
            "target_month"
        ] = future_date.month
    
    
    if "target_quarter" in future_row:
        future_row[
            "target_quarter"
        ] = future_date.quarter
    
    
    if "target_is_weekend" in future_row:
        future_row[
            "target_is_weekend"
        ] = int(
            future_date.dayofweek >= 5
        )
    
    
    # =====================================================
    # ONE-HOT ENCODED BLOOD GROUP
    # =====================================================
    
    blood_group_column = (
        f"blood_group_{selected_blood_group}"
    )
    
    if blood_group_column in future_row:
        future_row[blood_group_column] = 1
    
    
    # =====================================================
    # ONE-HOT ENCODED COMPONENT
    # =====================================================
    
    component_column = (
        f"component_{selected_component}"
    )
    
    if component_column in future_row:
        future_row[component_column] = 1
    
    
    # Convert to DataFrame
    X_future_day = pd.DataFrame(
        [future_row],
        columns=feature_columns
    )
    
    return X_future_day

In [15]:
# =========================================================
# RECURSIVE MULTI-DAY FORECAST
# =========================================================

def generate_future_forecast(
    model,
    product_history,
    selected_blood_group,
    selected_component,
    emergency_column,
    feature_columns,
    horizon_days=7
):
    
    # Sort historical data
    history = product_history.sort_values(
        "date"
    ).copy()
    
    
    # Historical demand values
    demand_history = (
        history["demand"]
        .astype(float)
        .tolist()
    )
    
    
    # Last historical date
    last_date = history["date"].max()
    
    
    # Future emergency values are unknown.
    # Prototype assumption:
    # Use the latest observed emergency value.
    latest_emergency = float(
        history[emergency_column].iloc[-1]
    )
    
    
    forecast_results = []
    
    
    # =====================================================
    # PREDICT EACH FUTURE DAY
    # =====================================================
    
    for day in range(1, horizon_days + 1):
        
        # Future date
        future_date = (
            last_date
            + pd.Timedelta(days=day)
        )
        
        
        # Build feature row
        X_future_day = build_future_feature_row(
            demand_history=demand_history,
            future_date=future_date,
            selected_blood_group=selected_blood_group,
            selected_component=selected_component,
            emergency_value=latest_emergency,
            feature_columns=feature_columns
        )
        
        
        # Generate prediction
        prediction = float(
            model.predict(X_future_day)[0]
        )
        
        
        # Demand cannot be negative
        prediction = max(0.0, prediction)
        
        
        # Store result
        forecast_results.append({
            "date": future_date,
            "blood_group": selected_blood_group,
            "component": selected_component,
            "predicted_demand": prediction
        })
        
        
        # Add prediction to history
        # for the next recursive forecast
        demand_history.append(prediction)
    
    
    return pd.DataFrame(forecast_results)

In [16]:
# =========================================================
# GENERATE 7-DAY FORECAST
# =========================================================

future_forecast = generate_future_forecast(
    model=model,
    product_history=product_history,
    selected_blood_group=SELECTED_BLOOD_GROUP,
    selected_component=SELECTED_COMPONENT,
    emergency_column=emergency_column,
    feature_columns=feature_columns,
    horizon_days=HORIZON_DAYS
)

print("7-Day Future Forecast:")

display(future_forecast)

7-Day Future Forecast:


,date,blood_group,component,predicted_demand
0,2024-12-31,O+,RBC,52.510723
1,2025-01-01,O+,RBC,55.671860
2,2025-01-02,O+,RBC,55.182060
3,2025-01-03,O+,RBC,55.539150
4,2025-01-04,O+,RBC,59.336002
5,2025-01-05,O+,RBC,62.161205
6,2025-01-06,O+,RBC,52.591389


In [17]:
# =========================================================
# TOTAL FORECAST DEMAND
# =========================================================

forecast_demand = float(
    future_forecast[
        "predicted_demand"
    ].sum()
)

print(
    f"Total predicted demand for the next "
    f"{HORIZON_DAYS} days:"
)

print(
    f"{forecast_demand:.2f} units"
)

Total predicted demand for the next 7 days:
392.99 units


In [18]:
# =========================================================
# MULTI-RISK INVENTORY ENGINE
# =========================================================

def assess_blood_inventory(
    current_stock,
    forecast_demand,
    safety_stock,
    near_expiry_units,
    expected_demand_before_expiry,
    shortage_threshold=SHORTAGE_HIGH_THRESHOLD,
    excess_threshold=EXCESS_HIGH_THRESHOLD,
    expiry_threshold=EXPIRY_HIGH_THRESHOLD
):
    
    # =====================================================
    # RISK 1: SHORTAGE
    # =====================================================
    
    projected_stock = (
        current_stock
        - forecast_demand
    )
    
    shortage_gap = max(
        0,
        forecast_demand
        + safety_stock
        - current_stock
    )
    
    if shortage_gap >= shortage_threshold:
        shortage_risk = "HIGH"
    
    elif shortage_gap > 0:
        shortage_risk = "MEDIUM"
    
    else:
        shortage_risk = "LOW"
    
    
    # =====================================================
    # RISK 2: EXCESS STOCK
    # =====================================================
    
    potential_excess = max(
        0,
        current_stock
        - forecast_demand
        - safety_stock
    )
    
    if potential_excess >= excess_threshold:
        excess_risk = "HIGH"
    
    elif potential_excess > 0:
        excess_risk = "MEDIUM"
    
    else:
        excess_risk = "LOW"
    
    
    # =====================================================
    # RISK 3: EXPIRY
    # =====================================================
    
    potential_unused = max(
        0,
        near_expiry_units
        - expected_demand_before_expiry
    )
    
    if potential_unused >= expiry_threshold:
        expiry_risk = "HIGH"
    
    elif potential_unused > 0:
        expiry_risk = "MEDIUM"
    
    else:
        expiry_risk = "LOW"
    
    
    # =====================================================
    # OVERALL RISK
    # =====================================================
    
    risks = [
        shortage_risk,
        excess_risk,
        expiry_risk
    ]
    
    if "HIGH" in risks:
        overall_risk = "HIGH"
    
    elif "MEDIUM" in risks:
        overall_risk = "MEDIUM"
    
    else:
        overall_risk = "LOW"
    
    
    # =====================================================
    # RETURN RESULTS
    # =====================================================
    
    return {
        "projected_stock":
            round(float(projected_stock), 2),
        
        "shortage_gap":
            round(float(shortage_gap), 2),
        
        "potential_excess":
            round(float(potential_excess), 2),
        
        "potential_unused_before_expiry":
            round(float(potential_unused), 2),
        
        "shortage_risk":
            shortage_risk,
        
        "excess_risk":
            excess_risk,
        
        "expiry_risk":
            expiry_risk,
        
        "overall_risk":
            overall_risk
    }

In [19]:
# =========================================================
# INVENTORY INPUTS
# =========================================================

current_stock = 100

safety_stock = 15

near_expiry_units = 20

expected_demand_before_expiry = 12


print("Current Stock:", current_stock)
print("Safety Stock:", safety_stock)
print("Near Expiry Units:", near_expiry_units)

Current Stock: 100
Safety Stock: 15
Near Expiry Units: 20


In [20]:
# =========================================================
# RUN MULTI-RISK ASSESSMENT
# =========================================================

risk_result = assess_blood_inventory(
    
    current_stock=current_stock,
    
    forecast_demand=forecast_demand,
    
    safety_stock=safety_stock,
    
    near_expiry_units=near_expiry_units,
    
    expected_demand_before_expiry=(
        expected_demand_before_expiry
    )
)

print(
    f"Risk Assessment for "
    f"{SELECTED_BLOOD_GROUP} "
    f"{SELECTED_COMPONENT}"
)

print("-" * 40)

for key, value in risk_result.items():
    
    print(
        f"{key}: {value}"
    )

Risk Assessment for O+ RBC
----------------------------------------
projected_stock: -292.99
shortage_gap: 307.99
potential_excess: 0.0
potential_unused_before_expiry: 8.0
shortage_risk: HIGH
excess_risk: LOW
expiry_risk: MEDIUM
overall_risk: HIGH


In [21]:
# =========================================================
# RISK EXPLANATIONS
# =========================================================

def generate_risk_explanations(risk_result):
    
    explanations = {}
    
    
    # Shortage explanation
    if risk_result["shortage_gap"] > 0:
        
        explanations["shortage"] = (
            "Forecast demand plus safety stock "
            "exceeds current available inventory."
        )
    
    else:
        
        explanations["shortage"] = (
            "No shortage gap was detected "
            "under the current rule."
        )
    
    
    # Excess explanation
    if risk_result["potential_excess"] > 0:
        
        explanations["excess"] = (
            "Current stock exceeds forecast demand "
            "and safety stock."
        )
    
    else:
        
        explanations["excess"] = (
            "No potential excess stock was detected."
        )
    
    
    # Expiry explanation
    if (
        risk_result[
            "potential_unused_before_expiry"
        ] > 0
    ):
        
        explanations["expiry"] = (
            "Some near-expiry units may remain "
            "unused before expiry."
        )
    
    else:
        
        explanations["expiry"] = (
            "No potential unused near-expiry units "
            "were detected."
        )
    
    
    return explanations

In [24]:
# =========================================================
# DASHBOARD-READY OUTPUT
# =========================================================

dashboard_output = {
    
    "blood_group":
        SELECTED_BLOOD_GROUP,
    
    "component":
        SELECTED_COMPONENT,
    
    "planning_horizon_days":
        HORIZON_DAYS,
    
    
    # Inventory information
    
    "current_stock":
        current_stock,
    
    "safety_stock":
        safety_stock,
    
    "near_expiry_units":
        near_expiry_units,
    
    
    # Forecast
    
    "forecast_demand":
        round(float(forecast_demand), 2),
    
    
    # Risk calculations
    
    **risk_result,
    
    
    # Explainability
    
    "explanations":
        generate_risk_explanations(
            risk_result
        )
}

print(
    json.dumps(
        dashboard_output,
        indent=4
    )
)



# This output is useful for your future backend API because it produces a structure like:

# {
#     "blood_group": "O+",
#     "component": "RBC",
#     "planning_horizon_days": 7,
#     "current_stock": 100,
#     "forecast_demand": 120.5,
#     "shortage_risk": "HIGH",
#     "excess_risk": "LOW",
#     "expiry_risk": "MEDIUM",
#     "overall_risk": "HIGH"
# }


{
    "blood_group": "O+",
    "component": "RBC",
    "planning_horizon_days": 7,
    "current_stock": 100,
    "safety_stock": 15,
    "near_expiry_units": 20,
    "forecast_demand": 392.99,
    "projected_stock": -292.99,
    "shortage_gap": 307.99,
    "potential_excess": 0.0,
    "potential_unused_before_expiry": 8.0,
    "shortage_risk": "HIGH",
    "excess_risk": "LOW",
    "expiry_risk": "MEDIUM",
    "overall_risk": "HIGH",
    "explanations": {
        "shortage": "Forecast demand plus safety stock exceeds current available inventory.",
        "excess": "No potential excess stock was detected.",
        "expiry": "Some near-expiry units may remain unused before expiry."
    }
}


In [23]:
# =========================================================
# TEST CASE
# =========================================================

test_result = assess_blood_inventory(
    
    current_stock=20,
    
    forecast_demand=35,
    
    safety_stock=8,
    
    near_expiry_units=50,
    
    expected_demand_before_expiry=20
)

print("Test Results:")

for key, value in test_result.items():
    
    print(
        f"{key}: {value}"
    )

Test Results:
projected_stock: -15.0
shortage_gap: 23.0
potential_excess: 0.0
potential_unused_before_expiry: 30.0
shortage_risk: HIGH
excess_risk: LOW
expiry_risk: HIGH
overall_risk: HIGH
